# Compute stability radius and max amplification

**Do not run this notebook** — it is included for reproducibility only. It shows how
`pseudospectrum_analysis.pkl` (loaded by `fig5.ipynb`) was generated.

For each matrix J in all three non-normal ensembles, it computes:
- **Stability radius** `r_stab = min_ω σ_min(jωI - M)` where `M = J - I`,
  using a coarse + refined grid search over ω.
- **Max resolvent amplification** `A_max = 1 / r_stab`.
- **Transient growth** `max_t ||exp(tM)||_2` computed via matrix exponential SVD.

Results are saved to `pseudospectrum_analysis.pkl` as a list of dicts,
one per matrix, in the order `matrices0 + matrices1 + matrices2`.

In [ ]:
import os
os.environ['OMP_NUM_THREADS']      = '1'
os.environ['MKL_NUM_THREADS']      = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pickle
from scipy.linalg import svdvals, expm
from joblib import Parallel, delayed
import sys
sys.path.append('../../..')
sys.path.append('../../../results')

In [ ]:
# --- Load data ---
# Download from Zenodo [DOI] and place under data/fig2/
data_root = '../../../data/fig2'
sigmas    = ['sigma02', 'sigma05', 'sigma1']

data = []
for s in sigmas:
    with open(f'{data_root}/{s}/Ws.pkl', 'rb') as f:
        data.append(pickle.load(f))

solutions = [[d for d in ds if not np.all(d.get('W') == 0)] for ds in data]
matrices  = [[d['W']      for d in s] for s in solutions]
lambdas   = [[d['Lambda'] for d in s] for s in solutions]

all_Js   = matrices[0] + matrices[1] + matrices[2]
all_eigs = lambdas[0]  + lambdas[1]  + lambdas[2]
print(f'Total matrices: {len(all_Js)}')

In [ ]:
# --- Analysis functions ---
def J_to_M(J):
    return J - np.eye(J.shape[0])

def _sigma_min_jwI_minus_M(M, w):
    """Smallest singular value of (jωI - M)."""
    A = 1j * w * np.eye(M.shape[0], dtype=complex) - M
    return svdvals(A)[-1]

def stability_radius_coarse_refine(M, omega_min=-5.0, omega_max=5.0,
                                   n_coarse=200, n_refine=400,
                                   refine_half_width=0.5):
    """
    Approximate r_stab = min_ω σ_min(jωI - M) and A_max = 1/r_stab
    using a coarse grid search followed by a local refined search.
    """
    omegas_c  = np.linspace(omega_min, omega_max, n_coarse)
    sigmins_c = np.array([_sigma_min_jwI_minus_M(M, w) for w in omegas_c])

    w_c  = float(omegas_c[np.argmin(sigmins_c)])
    w_lo = max(omega_min, w_c - refine_half_width)
    w_hi = min(omega_max, w_c + refine_half_width)

    omegas_f  = np.linspace(w_lo, w_hi, n_refine)
    sigmins_f = np.array([_sigma_min_jwI_minus_M(M, w) for w in omegas_f])

    idx_f   = int(np.argmin(sigmins_f))
    r_stab  = float(sigmins_f[idx_f])
    w_star  = float(omegas_f[idx_f])
    A_max   = float(1.0 / r_stab) if r_stab > 0 else np.inf

    return r_stab, w_star, A_max, (omegas_c, sigmins_c), (omegas_f, sigmins_f)

def transient_growth_from_M(M, evals_J, t_max=None, n_times=200):
    """
    Compute max_t ||exp(tM)||_2 over t in [0, t_max].
    t_max defaults to 8 / |max Re(λ(M))|.
    """
    alpha_max_M = float(np.max(np.asarray(evals_J).real - 1.0))
    if alpha_max_M >= 0:
        raise ValueError('M = J - I appears unstable (max Re λ ≥ 0).')
    if t_max is None:
        t_max = 8.0 / abs(alpha_max_M)

    ts = np.linspace(0.0, t_max, n_times)
    G  = np.array([svdvals(expm(t * M))[0] for t in ts])

    idx_max = int(np.argmax(G))
    return ts, G, float(G[idx_max]), float(ts[idx_max])

def analyze_single_J_fast(J, evals_J,
                           omega_min=-5.0, omega_max=5.0,
                           n_coarse=200, n_refine=400,
                           refine_half_width=0.5,
                           t_max=None, n_times=200,
                           store_curves=False):
    M = J_to_M(J)
    evals_J = np.asarray(evals_J)

    alpha_J = float(np.max(evals_J.real))
    alpha_M = float(np.max(evals_J.real - 1.0))

    r_stab, omega_star, A_max, coarse_pack, refine_pack = stability_radius_coarse_refine(
        M, omega_min=omega_min, omega_max=omega_max,
        n_coarse=n_coarse, n_refine=n_refine,
        refine_half_width=refine_half_width)

    ts, G, G_max, t_star = transient_growth_from_M(
        M, evals_J, t_max=t_max, n_times=n_times)

    out = dict(
        alpha_J=alpha_J, alpha_M=alpha_M,
        r_stab=r_stab, omega_star=omega_star, A_max=A_max,
        G_max=G_max, t_star=t_star,
        t_max=float(ts[-1]),
    )
    if store_curves:
        out['resolvent_coarse'] = dict(omegas=coarse_pack[0], sigmin=coarse_pack[1])
        out['resolvent_refine'] = dict(omegas=refine_pack[0], sigmin=refine_pack[1])
        out['transient']        = dict(ts=ts, G=G)
    return out

In [ ]:
# --- Run in parallel ---
# WARNING: takes a long time
results_all = Parallel(n_jobs=6, verbose=10)(
    delayed(analyze_single_J_fast)(
        J, evals_J,
        omega_min=-5.0, omega_max=5.0,
        n_coarse=200, n_refine=400,
        refine_half_width=0.5,
        t_max=None, n_times=200
    )
    for J, evals_J in zip(all_Js, all_eigs)
)

In [ ]:
# --- Save ---
with open('../../../data/fig5/pseudospectrum_analysis.pkl', 'wb') as f:
    pickle.dump(results_all, f)
print(f'Saved {len(results_all)} records.')